# Swiggy Delivery Time Prediction


### Business Problem


-  Swiggy processes millions of food-delivery journeys across hundreds of cities every month. Accurate Estimated Time of Arrival (ETA) at order placement and during delivery is core to the customer experience — it affects conversion, cancellations, ratings, and retention. Today’s ETA accuracy suffers from many sources of uncertainty: highly variable traffic patterns, weather events, restaurant preparation variability, rider behaviour and skill, order batching/multiple deliveries, and city-specific operational constraints. Inaccurate ETAs cause:
Reduced customer trust and lower repeat orders.


- Increased cancellations and refund costs.


- Higher support volume and operational interventions.


- Sub-optimal rider allocation and increased rider idle or overtime costs.


### Problem Statement

- Swiggy’s objective is to predict the delivery time (minutes) per order at the moment of order placement (and update it dynamically), with business-grade accuracy and uncertainty estimates so the platform can
  
 - (a) show reliable ETAs to customers
 - (b) optimize rider allocation.

   
- Build a production-ready, scalable Machine Learning system that predicts per-order delivery time (in minutes) using real-time and historical features (order, rider, restaurant, geospatial, traffic, weather, and temporal signals). The system must produce:


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\Praneeth\Downloads\swiggy_demographic.csv")

In [3]:
df

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,time_taken,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,24,INDO,19,3,saturday,1,15.0,11.0,morning,3.025149
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,33,BANG,25,3,friday,0,5.0,19.0,evening,20.183530
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,26,BANG,19,3,saturday,1,15.0,8.0,morning,1.552758
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,21,COIMB,5,4,tuesday,0,10.0,18.0,evening,7.790401
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,30,CHEN,26,3,saturday,1,15.0,13.0,afternoon,6.210138
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,JAPRES04DEL01,30.0,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24,windy,high,...,32,JAP,24,3,thursday,0,10.0,11.0,morning,1.489846
45498,AGRRES16DEL01,21.0,4.6,NaN,NaN,NaN,NaN,2022-02-16,windy,jam,...,36,AGR,16,2,wednesday,0,15.0,19.0,evening,NaN
45499,CHENRES08DEL03,30.0,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11,cloudy,low,...,16,CHEN,11,3,friday,0,15.0,23.0,night,4.657195
45500,COIMBRES11DEL01,20.0,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07,cloudy,high,...,26,COIMB,7,3,monday,0,5.0,13.0,afternoon,6.232393


In [4]:
df.shape

(45502, 26)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45502 entries, 0 to 45501
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   rider_id              45502 non-null  object 
 1   age                   43648 non-null  float64
 2   ratings               43594 non-null  float64
 3   restaurant_latitude   41872 non-null  float64
 4   restaurant_longitude  41872 non-null  float64
 5   delivery_latitude     41872 non-null  float64
 6   delivery_longitude    41872 non-null  float64
 7   order_date            45502 non-null  object 
 8   weather               44977 non-null  object 
 9   traffic               44992 non-null  object 
 10  vehicle_condition     45502 non-null  int64  
 11  type_of_order         45502 non-null  object 
 12  type_of_vehicle       45502 non-null  object 
 13  multiple_deliveries   44509 non-null  float64
 14  festival              45274 non-null  object 
 15  city_type          

In [6]:
df.describe()

,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,vehicle_condition,multiple_deliveries,time_taken,order_day,order_month,is_weekend,pickup_time_minutes,order_time_hour,distance
count,43648.000000,43594.000000,41872.000000,41872.000000,41872.000000,41872.000000,45502.000000,44509.000000,45502.000000,45502.000000,45502.000000,45502.000000,43862.000000,43862.000000,41872.000000
mean,29.555008,4.635287,18.913696,76.921664,18.977356,76.985325,1.019406,0.744928,26.297591,13.811657,2.980726,0.274867,9.989399,17.423966,9.719296
std,5.761482,0.313827,5.467265,3.503107,5.469056,3.503260,0.835229,0.572488,9.386419,8.709540,0.546031,0.446452,4.087516,4.817856,5.602890
min,20.000000,2.500000,9.957144,72.768726,9.967144,72.778726,0.000000,0.000000,10.000000,1.000000,2.000000,0.000000,5.000000,0.000000,1.465067
25%,25.000000,4.500000,12.986047,73.897902,13.065996,73.940327,0.000000,0.000000,19.000000,6.000000,3.000000,0.000000,5.000000,15.000000,4.657655
50%,30.000000,4.700000,19.065838,76.618203,19.124049,76.662620,1.000000,1.000000,26.000000,13.000000,3.000000,0.000000,10.000000,19.000000,9.193014
75%,35.000000,4.900000,22.751234,78.368855,22.820040,78.405467,2.000000,1.000000,32.000000,20.000000,3.000000,1.000000,15.000000,21.000000,13.680920
max,39.000000,5.000000,30.914057,88.433452,31.054057,88.563452,3.000000,3.000000,54.000000,31.000000,4.000000,1.000000,15.000000,23.000000,20.969489


In [7]:
df.columns

Index(['rider_id', 'age', 'ratings', 'restaurant_latitude',
       'restaurant_longitude', 'delivery_latitude', 'delivery_longitude',
       'order_date', 'weather', 'traffic', 'vehicle_condition',
       'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival',
       'city_type', 'time_taken', 'city_name', 'order_day', 'order_month',
       'order_day_of_week', 'is_weekend', 'pickup_time_minutes',
       'order_time_hour', 'order_time_of_day', 'distance'],
      dtype='object')

In [8]:
df.isna().sum()

rider_id                   0
age                     1854
ratings                 1908
restaurant_latitude     3630
restaurant_longitude    3630
delivery_latitude       3630
delivery_longitude      3630
order_date                 0
weather                  525
traffic                  510
vehicle_condition          0
type_of_order              0
type_of_vehicle            0
multiple_deliveries      993
festival                 228
city_type               1198
time_taken                 0
city_name                  0
order_day                  0
order_month                0
order_day_of_week          0
is_weekend                 0
pickup_time_minutes     1640
order_time_hour         1640
order_time_of_day          0
distance                3630
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

###  Data Preprocessing

In [10]:
df

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,time_taken,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,24,INDO,19,3,saturday,1,15.0,11.0,morning,3.025149
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,33,BANG,25,3,friday,0,5.0,19.0,evening,20.183530
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,26,BANG,19,3,saturday,1,15.0,8.0,morning,1.552758
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,21,COIMB,5,4,tuesday,0,10.0,18.0,evening,7.790401
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,30,CHEN,26,3,saturday,1,15.0,13.0,afternoon,6.210138
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,JAPRES04DEL01,30.0,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24,windy,high,...,32,JAP,24,3,thursday,0,10.0,11.0,morning,1.489846
45498,AGRRES16DEL01,21.0,4.6,NaN,NaN,NaN,NaN,2022-02-16,windy,jam,...,36,AGR,16,2,wednesday,0,15.0,19.0,evening,NaN
45499,CHENRES08DEL03,30.0,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11,cloudy,low,...,16,CHEN,11,3,friday,0,15.0,23.0,night,4.657195
45500,COIMBRES11DEL01,20.0,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07,cloudy,high,...,26,COIMB,7,3,monday,0,5.0,13.0,afternoon,6.232393


In [11]:
X = df.drop(columns=['rider_id','order_date','restaurant_latitude','restaurant_longitude','delivery_latitude','delivery_longitude'],inplace=True)

In [12]:
important_cols = [
 'age',
 'ratings',
 'distance',
 'pickup_time_minutes',
 'weather',
 'traffic',
 'vehicle_condition',
 'type_of_order',
 'type_of_vehicle',
 'multiple_deliveries',
 'festival',
 'city_type',
 'order_day_of_week',
 'is_weekend',
 'order_time_hour'
]

In [13]:
# Separate Input and Output Data
X = df[important_cols]
y = df['time_taken']

In [14]:
X

,age,ratings,distance,pickup_time_minutes,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,order_day_of_week,is_weekend,order_time_hour
0,37.0,4.9,3.025149,15.0,sunny,high,2,snack,motorcycle,0.0,no,urban,saturday,1,11.0
1,34.0,4.5,20.183530,5.0,stormy,jam,2,snack,scooter,1.0,no,metropolitian,friday,0,19.0
2,23.0,4.4,1.552758,15.0,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,saturday,1,8.0
3,38.0,4.7,7.790401,10.0,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,tuesday,0,18.0
4,32.0,4.6,6.210138,15.0,cloudy,high,1,snack,scooter,1.0,no,metropolitian,saturday,1,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,1.489846,10.0,windy,high,1,meal,motorcycle,0.0,no,metropolitian,thursday,0,11.0
45498,21.0,4.6,NaN,15.0,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,wednesday,0,19.0
45499,30.0,4.9,4.657195,15.0,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,friday,0,23.0
45500,20.0,4.7,6.232393,5.0,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,monday,0,13.0


In [15]:
y

0        24
1        33
2        26
3        21
4        30
         ..
45497    32
45498    36
45499    16
45500    26
45501    36
Name: time_taken, Length: 45502, dtype: int64

In [16]:
# Seggregate the training and testing

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=23)

In [17]:
df[important_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45502 entries, 0 to 45501
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  43648 non-null  float64
 1   ratings              43594 non-null  float64
 2   distance             41872 non-null  float64
 3   pickup_time_minutes  43862 non-null  float64
 4   weather              44977 non-null  object 
 5   traffic              44992 non-null  object 
 6   vehicle_condition    45502 non-null  int64  
 7   type_of_order        45502 non-null  object 
 8   type_of_vehicle      45502 non-null  object 
 9   multiple_deliveries  44509 non-null  float64
 10  festival             45274 non-null  object 
 11  city_type            44304 non-null  object 
 12  order_day_of_week    45502 non-null  object 
 13  is_weekend           45502 non-null  int64  
 14  order_time_hour      43862 non-null  float64
dtypes: float64(6), int64(2), object(7)
m

In [18]:
X

,age,ratings,distance,pickup_time_minutes,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,order_day_of_week,is_weekend,order_time_hour
0,37.0,4.9,3.025149,15.0,sunny,high,2,snack,motorcycle,0.0,no,urban,saturday,1,11.0
1,34.0,4.5,20.183530,5.0,stormy,jam,2,snack,scooter,1.0,no,metropolitian,friday,0,19.0
2,23.0,4.4,1.552758,15.0,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,saturday,1,8.0
3,38.0,4.7,7.790401,10.0,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,tuesday,0,18.0
4,32.0,4.6,6.210138,15.0,cloudy,high,1,snack,scooter,1.0,no,metropolitian,saturday,1,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,1.489846,10.0,windy,high,1,meal,motorcycle,0.0,no,metropolitian,thursday,0,11.0
45498,21.0,4.6,NaN,15.0,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,wednesday,0,19.0
45499,30.0,4.9,4.657195,15.0,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,friday,0,23.0
45500,20.0,4.7,6.232393,5.0,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,monday,0,13.0


In [19]:
# Seggregating the columns based on data types

Nominal_cols = ['weather','type_of_order','type_of_vehicle','city_type','festival','order_day_of_week']

Ordinal_cols = ['traffic']

Numerical_cols = ['age','ratings','distance','pickup_time_minutes','vehicle_condition','multiple_deliveries','is_weekend','order_time_hour']

In [20]:
df['multiple_deliveries'].unique()

array([ 0.,  1.,  3., nan,  2.])

In [21]:
df['vehicle_condition'].unique()

array([2, 0, 1, 3])

In [22]:
df['festival'].unique()

array(['no', 'yes', nan], dtype=object)

In [23]:
df['order_day_of_week'].unique()

array(['saturday', 'friday', 'tuesday', 'monday', 'sunday', 'wednesday',
       'thursday'], dtype=object)

In [24]:
df['is_weekend'].unique()

array([1, 0])

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [26]:
nominal_pipeline=Pipeline([('Mode Imputer',SimpleImputer(strategy='most_frequent')),
                            ('OneHotEncoder',OneHotEncoder(drop='first',dtype='int',handle_unknown='ignore'))])
nominal_pipeline

,steps,"[('Mode Imputer', ...), ('OneHotEncoder', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,'auto'


In [27]:
df['traffic'].unique()

array(['high', 'jam', 'low', 'medium', nan], dtype=object)

In [28]:
order = [['low','medium','high','jam']]

In [29]:
ordinal_pipeline=Pipeline([('Mode Imputer',SimpleImputer(strategy='most_frequent')),
                           ('Ordinal Encoding',OrdinalEncoder(categories=order))])
ordinal_pipeline

,steps,"[('Mode Imputer', ...), ('Ordinal Encoding', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,categories,"[['low', 'medium', ...]]"


In [30]:
df[important_cols].describe()

,age,ratings,distance,pickup_time_minutes,vehicle_condition,multiple_deliveries,is_weekend,order_time_hour
count,43648.000000,43594.000000,41872.000000,43862.000000,45502.000000,44509.000000,45502.000000,43862.000000
mean,29.555008,4.635287,9.719296,9.989399,1.019406,0.744928,0.274867,17.423966
std,5.761482,0.313827,5.602890,4.087516,0.835229,0.572488,0.446452,4.817856
min,20.000000,2.500000,1.465067,5.000000,0.000000,0.000000,0.000000,0.000000
25%,25.000000,4.500000,4.657655,5.000000,0.000000,0.000000,0.000000,15.000000
50%,30.000000,4.700000,9.193014,10.000000,1.000000,1.000000,0.000000,19.000000
75%,35.000000,4.900000,13.680920,15.000000,2.000000,1.000000,1.000000,21.000000
max,39.000000,5.000000,20.969489,15.000000,3.000000,3.000000,1.000000,23.000000


In [31]:
from sklearn.preprocessing import StandardScaler

In [32]:
numerical_pipeline=Pipeline([('Median Imputation',SimpleImputer(strategy='median')),
                             ('Scaling',StandardScaler())])
numerical_pipeline

,steps,"[('Median Imputation', ...), ('Scaling', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [33]:
from sklearn.compose import ColumnTransformer

In [34]:
Data_preprocessing=ColumnTransformer([('Nomial transformation',nominal_pipeline,Nominal_cols),
                   ('Ordinal transformation',ordinal_pipeline,Ordinal_cols),
                   ('Numerical transdormation',numerical_pipeline,Numerical_cols)]) 
Data_preprocessing

,transformers,"[('Nomial transformation', ...), ('Ordinal transformation', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'most_frequent'
,fill_value,None


In [35]:
Data_preprocessing.fit_transform(X)

array([[ 0.        ,  0.        ,  0.        , ..., -1.32263053,
         1.62422986, -1.36745444],
       [ 0.        ,  0.        ,  1.        , ...,  0.43971491,
        -0.6156764 ,  0.32056032],
       [ 0.        ,  1.        ,  0.        , ...,  0.43971491,
         1.62422986, -2.00045998],
       ...,
       [ 0.        ,  0.        ,  0.        , ..., -1.32263053,
        -0.6156764 ,  1.1645677 ],
       [ 0.        ,  0.        ,  0.        , ...,  0.43971491,
        -0.6156764 , -0.94545075],
       [ 1.        ,  0.        ,  0.        , ...,  0.43971491,
        -0.6156764 , -0.10144337]], shape=(45502, 29))

In [36]:
import numpy as np
np.isnan(Data_preprocessing.fit_transform(X)).sum()

np.int64(0)

### MODEL BUILDING

### Linear Regression

In [37]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error

In [38]:
lr_model=Pipeline([('Data Preprocessing',Data_preprocessing),
          ('Algorithm',LinearRegression())])
lr_model.fit(X_train,y_train)
y_pred= lr_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")

R2 Score: 0.5894794654207829
MAE: 4.844420076585154


### KNN for Regression

In [39]:
from sklearn.neighbors import KNeighborsRegressor

In [40]:
knn_model=Pipeline([('Data Preprocessing',Data_preprocessing),
                    ('Algorithm',KNeighborsRegressor())])
knn_model.fit(X_train,y_train)
y_pred=knn_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")

R2 Score: 0.6368984374256206
MAE: 4.448368311174597


### Decision Tree 

In [41]:
from sklearn.tree import DecisionTreeRegressor

In [42]:
dt_model=Pipeline([('Data Preprocessing',Data_preprocessing),
                    ('Algorithm',DecisionTreeRegressor())])
dt_model.fit(X_train,y_train)
y_pred=dt_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")

R2 Score: 0.6244793302875736
MAE: 4.370398857268432


### RandomForest

In [43]:
from sklearn.ensemble import RandomForestRegressor

In [44]:
rf_model=Pipeline([('Data Preprocessing',Data_preprocessing),
                    ('Algorithm',RandomForestRegressor())])
rf_model.fit(X_train,y_train)
y_pred=rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")


R2 Score: 0.8043782881426795
MAE: 3.283879793429294


In [45]:
X_trainning = rf_model.predict(X_train)
X_testing = rf_model.predict(X_test)

train_r2 = r2_score(y_train,X_trainning)
test_r2 = r2_score(y_test,X_testing)

print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")

Train R2: 0.9722914412646346
Test R2: 0.8043782881426795


### SVR

In [46]:
from sklearn.svm import SVR

In [47]:
svr_model=Pipeline([('Data Preprocessing',Data_preprocessing),
                    ('Algorithm',SVR())])
svr_model.fit(X_train,y_train)
y_pred=svr_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")

R2 Score: 0.7187497490508049
MAE: 3.9507798405806946


In [48]:
X_trainning = svr_model.predict(X_train)
X_testing = svr_model.predict(X_test)

train_r2 = r2_score(y_train,X_trainning)
test_r2 = r2_score(y_test,X_testing)

print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")

Train R2: 0.7207625781088062
Test R2: 0.7187497490508049


### XGBRegressor

In [52]:
from xgboost import XGBRegressor

In [53]:
xgb_model=Pipeline([('Data Preprocessing',Data_preprocessing),
                    ('Algorithm',XGBRegressor())])
xgb_model.fit(X_train,y_train)
y_pred=xgb_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")

R2 Score: 0.8091675043106079
MAE: 3.2738077640533447


In [54]:
X_trainning = xgb_model.predict(X_train)
X_testing = xgb_model.predict(X_test)

train_r2 = r2_score(y_train,X_trainning)
test_r2 = r2_score(y_test,X_testing)

print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")

Train R2: 0.8527177572250366
Test R2: 0.8091675043106079


### Cross  Validation

In [58]:
from sklearn.model_selection import KFold,cross_val_score

In [59]:
kf = KFold(n_splits=5,shuffle=True,random_state=23)
cross_val_score(xgb_model,X,y,scoring='r2',cv=kf).mean()

np.float64(0.8053919792175293)

### HyperParameter Tuning

In [62]:
from sklearn.model_selection import RandomizedSearchCV

In [76]:
param_dist = {
    'Algorithm__n_estimators': [100, 200, 300],
    'Algorithm__max_depth': [3, 5, 7],
    'Algorithm__learning_rate': [0.01, 0.1, 0.2],
    'Algorithm__subsample': [0.8, 1.0],
    'Algorithm__colsample_bytree': [0.8, 1.0]
}

In [77]:
rsv = RandomizedSearchCV(estimator=xgb_model,param_distributions=param_dist,cv=kf,scoring='r2',random_state=23)

In [78]:
rsv.fit(X,y)

,estimator,"Pipeline(step...=None, ...))])"
,param_distributions,"{'Algorithm__colsample_bytree': [0.8, 1.0], 'Algorithm__learning_rate': [0.01, 0.1, ...], 'Algorithm__max_depth': [3, 5, ...], 'Algorithm__n_estimators': [100, 200, ...], ...}"
,n_iter,10
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,23
,error_score,nan


In [79]:
rsv.best_params_

{'Algorithm__subsample': 0.8,
 'Algorithm__n_estimators': 200,
 'Algorithm__max_depth': 7,
 'Algorithm__learning_rate': 0.1,
 'Algorithm__colsample_bytree': 1.0}

In [80]:
rsv.best_score_

np.float64(0.8092320442199707)

## Final Model

In [83]:
final_model=Pipeline([('Data Preprocessing',Data_preprocessing),
                    ('Algorithm',XGBRegressor(subsample=0.8,n_estimators=200,max_depth=7,learning_rate=0.1,colsample_bytree=1.0))])

In [85]:
y_pred = final_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R2 Score: {r2}")
print(f"MAE: {mae}")

R2 Score: 0.8665465712547302
MAE: 2.72697114944458


In [86]:
final_model.fit(X,y)

,steps,"[('Data Preprocessing', ...), ('Algorithm', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Nomial transformation', ...), ('Ordinal transformation', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [88]:
import pickle
with open(r'D:\Swiggy Delivery Time Prediction Project.pkl','wb') as f:
    pickle.dump((final_model),f)

In [89]:
import pickle

with open(r'D:\Swiggy Delivery Time Prediction Project.pkl', "rb") as f:
    final_model = pickle.load(f)